In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import ast

# matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings

warnings.filterwarnings("ignore")

# 폰트 설정(없으면 영문으로 나옴)
try:
    font_path = [f for f in fm.findSystemFonts() if "NanumGothic" in f or "Malgun" in f or "AppleGothic" in f]
    if font_path:
        plt.rcParams["font.family"] = fm.FontProperties(fname=font_path[0]).get_name()
    else:
        plt.rcParams["font.family"] = "DejaVu Sans"
except:
    plt.rcParams["font.family"] = "DejaVu Sans"

plt.rcParams["axes.unicode_minus"] = False

In [ ]:
funnel = pd.read_csv("../pjh_EDA/funnel_instance.csv")
full = pd.read_csv("../pjh_EDA/preprocessed_final.csv")

# is_aware 재생성 (CSV에 없으므로)
funnel["is_aware"] = (
    (funnel["was_viewed"] == 1) & (funnel["was_completed"] == 1) & (funnel["completed_without_view"] == 0).astype(int)
)

# 오퍼 속성 병합
portfolio = (
    full[
        [
            "offer_id",
            "difficulty",
            "reward",
            "duration",
            "offer_type",
            "ch_email",
            "ch_web",
            "ch_mobile",
            "ch_social",
            "channel_count",
        ]
    ]
    .dropna(subset=["offer_id"])
    .drop_duplicates("offer_id")
)
funnel = funnel.merge(portfolio, on="offer_id", how="left", suffixes=("", "_port"))

# 고객 속성 병합
cust = full[["customer_id", "gender", "age_group", "income_group", "income"]].drop_duplicates("customer_id")
funnel = funnel.merge(cust, on="customer_id", how="left")

# 분석 대상: informational 제외, 누락 제거
funnel_clean = funnel[
    funnel["offer_type"].isin(["bogo", "discount"])
    & (funnel["income_group"] != "누락")
    & (funnel["age_group"] != "누락")
].copy()

### | 채널 수 - 오퍼 타입별 전환율

In [ ]:
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

# 채널 수별 완료율 + 신뢰구간
for ch_cnt in sorted(funnel_clean["channel_count"].dropna().unique()):
    sub = funnel_clean[funnel_clean["channel_count"] == ch_cnt]
    n = len(sub)
    k = sub["was_completed"].sum()
    rate = k / n * 100
    lo, hi = proportion_confint(k, n, alpha=0.05, method="wilson")
    print(f"채널 {int(ch_cnt)}개  완료율 {rate:.1f}%  95%CI [{lo*100:.1f}%, {hi*100:.1f}%]  (n={n:,})")

### | 오퍼 속성(reward, difficulty)과 완료율 상관관계

In [ ]:
# 오퍼별 완료율 집계
offer_cr = (
    funnel_clean.groupby("offer_id")
    .agg(
        offer_type=("offer_type", "first"),
        reward=("reward", "first"),
        difficulty=("difficulty", "first"),
        channel_count=("channel_count", "first"),
        n=("was_completed", "count"),
        completed=("was_completed", "sum"),
    )
    .reset_index()
)
offer_cr["completion_rate"] = offer_cr["completed"] / offer_cr["n"] * 100
offer_cr["aware_rate"] = funnel_clean.groupby("offer_id")["is_aware"].mean().values * 100

print(
    offer_cr[["offer_id", "offer_type", "reward", "difficulty", "channel_count", "completion_rate", "aware_rate"]]
    .sort_values("completion_rate", ascending=False)
    .to_string(index=False)
)